#Cell 1: install. I copied this from Unsloth's current Qwen3-4B-Instruct notebook.

In [ ]:
%%capture
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
!unsloth install-kernels
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2

#Cell 2: load the model and LoRA

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Qwen3-4B-Instruct-2507", max_seq_length=8192, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32, lora_dropout=0, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407)
tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1120 01:09:46.923000 2986 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1120 01:09:47.034000 2986 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.7: Fast Qwen3 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2026.11.7 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen3DecoderLayer).


#Cell 3: data. The assert is the check that the tool-call template renders.

In [ ]:
import json
from datasets import Dataset
rows = [json.loads(l) for l in open("train.jsonl")]
texts = [tokenizer.apply_chat_template(r["messages"], tools=r["tools"], tokenize=False) for r in rows]
assert all("<tool_call>" in t and "<tool_response>" in t for t in texts), "template dropped tool turns"
print(len(texts), max(len(tokenizer(t).input_ids) for t in texts), "max tokens")
print(texts[0][:1500])
ds = Dataset.from_dict({"text": texts})

360 7568 max tokens
<|im_start|>system
You are a coding agent. You have access to a bash tool for running shell commands. You also have an edit tool for single-occurrence string replacement in a file.

External network access is disabled.

Work like this: first inspect the working directory and any provided source, and run the provided program on the example input and a few inputs of your own. Then write your solution to the deliverable file. Then check it against the example and your own inputs, and fix and repeat until it matches. A reply without a tool call ends the session immediately, so keep calling tools until the deliverable file is written and verified; reply in plain text only when you are finished.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "bash", "description": "Run a bash command and return its combined stdout and s

#Cell 4: train

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds, args=SFTConfig(
    dataset_text_field="text", max_seq_length=8192,
    per_device_train_batch_size=1, gradient_accumulation_steps=8,
    num_train_epochs=2, learning_rate=1e-4, warmup_steps=5, logging_steps=1,
    optim="adamw_8bit", weight_decay=0.001, lr_scheduler_type="linear",
    seed=3407, report_to="none",
    output_dir="/content/drive/MyDrive/re_ckpt_v3",   # Drive must be mounted; use a new name per run
    save_strategy="epoch", save_total_limit=3))
trainer = train_on_responses_only(trainer,
    instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/360 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=2):   0%|          | 0/360 [00:00<?, ? examples/s]

Unsloth: Packing is on by default - documents are packed into full rows with no cross-document attention. Set packing = False in SFTConfig (or UNSLOTH_DEFAULT_PACKING=0) to turn it off.


Map:   0%|          | 0/174 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 174 | Num Epochs = 2 | Total steps = 44
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.201400
2,1.221700
3,1.164800
4,1.076000
5,0.907700
6,0.820100
7,0.739100
8,0.716800
9,0.717900
10,0.696500


TrainOutput(global_step=44, training_loss=0.6304969408295371, metrics={'train_runtime': 5555.9882, 'train_samples_per_second': 0.063, 'train_steps_per_second': 0.008, 'total_flos': 6.029830659704218e+16, 'train_loss': 0.6304969408295371, 'epoch': 2.0})

Save the adapter to drive:

In [ ]:
model.save_pretrained("/content/drive/MyDrive/re_lora_v3")
tokenizer.save_pretrained("/content/drive/MyDrive/re_lora_v3")

('/content/drive/MyDrive/re_lora/tokenizer_config.json',
 '/content/drive/MyDrive/re_lora/special_tokens_map.json',
 '/content/drive/MyDrive/re_lora/chat_template.jinja',
 '/content/drive/MyDrive/re_lora/vocab.json',
 '/content/drive/MyDrive/re_lora/merges.txt',
 '/content/drive/MyDrive/re_lora/added_tokens.json',
 '/content/drive/MyDrive/re_lora/tokenizer.json')

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained("/content/drive/MyDrive/re_lora_v3_v2c", max_seq_length=8192, load_in_4bit=True)
model.save_pretrained_gguf("/content/drive/MyDrive/re_gguf_v3", tokenizer, quantization_method="q4_k_m")

W1120 05:57:14.428000 1282 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1120 05:57:14.488000 1282 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth 2026.11.7: no GPU found, training on CPU (x86_64, 1 threads, bfloat16 = False). CPU training is much slower than GPU training; supported: text LoRA / full finetuning with SFTTrainer. Set UNSLOTH_DISABLE_CPU_FALLBACK=1 to raise instead.


HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/content/drive/MyDrive/re_lora_v3_v2c'. Use `repo_type` argument if needed.

Cell 5: export the GGUF and download it.

In [ ]:
model.save_pretrained_gguf("re_sft", tokenizer, quantization_method="q4_k_m")
from google.colab import files
import glob; files.download(glob.glob("re_sft*/*.gguf")[0])

Unsloth: Merging model weights to 16-bit format...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files:  50%|█████     | 1/2 [05:17<05:17, 317.02s/it]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.08G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [09:13<00:00, 276.76s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit:  50%|█████     | 1/2 [04:37<04:37, 277.70s/it]

In [ ]:
!ls -la /content/re_sft_gguf

total 2438768
drwxr-xr-x 2 root root       4096 Nov 18 22:45 .
drwxr-xr-x 1 root root       4096 Nov 18 22:29 ..
-rw-r--r-- 1 root root       1550 Nov 18 22:45 Modelfile
-rw-r--r-- 1 root root 2497280416 Nov 18 22:45 Qwen3-4B-Instruct-2507.Q4_K_M.gguf


#If bad runs were saved to Google Drive - delete with this cell

In [ ]:
import os, shutil, glob
D = "/content/drive/MyDrive"
targets = [f"{D}/re_lora"] + glob.glob(f"{D}/Qwen3-4B-Instruct-2507*.gguf") + [f"{D}/Modelfile"]
targets = [t for t in targets if os.path.exists(t)]
for t in targets:
    print(t, "(folder)" if os.path.isdir(t) else f"({os.path.getsize(t)/1e9:.2f} GB)")
if targets and input("Delete the above? type yes: ") == "yes":
    for t in targets:
        shutil.rmtree(t) if os.path.isdir(t) else os.remove(t)
    print("deleted")
else:
    print("nothing deleted")

/content/drive/MyDrive/re_lora (folder)
Delete the above? type yes: yes
deleted
